# Bitcoin Naive Forecast Audit

## Role
Final proof of the rolling persistence implementation.

## Inputs
Canonical target and frozen validated forecasts.

## Outputs
Executed row audit, manual metrics, and PASS table.

## Depends On
01 and 07.

## Authoritative Status
AUTHORITATIVE VALIDATION

## What This Notebook Does Not Do
It does not compare historical models or write forecasts.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.bitcoin_pipeline import *
RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False


In [2]:
# Load the canonical split and frozen forecast matrix.
_, target = load_bitcoin_target(ROOT)
train, test = canonical_split(target)
validated = load_validated_forecasts(ROOT)

# Reconstruct persistence directly from the previous observed actual.
naive = target.shift(1).reindex(test.index)
rows = pd.DataFrame({
    'Forecast date': test.index,
    'Previous actual': [train.iloc[-1], *test.iloc[:-1]],
    'Manual naive': naive.to_numpy(),
    'Frozen naive': validated.Naive.to_numpy(),
})

# Display the existing leading and trailing row audit.
display(rows.head(10))
display(rows.tail(5))

,Forecast date,Previous actual,Manual naive,Frozen naive
0,2023-08-12 00:00:00+00:00,29398.0,29398.0,29398.0
1,2023-08-13 00:00:00+00:00,29415.0,29415.0,29415.0
2,2023-08-14 00:00:00+00:00,29284.0,29284.0,29284.0
3,2023-08-15 00:00:00+00:00,29408.0,29408.0,29408.0
4,2023-08-16 00:00:00+00:00,29172.0,29172.0,29172.0
5,2023-08-17 00:00:00+00:00,28701.0,28701.0,28701.0
6,2023-08-18 00:00:00+00:00,26642.0,26642.0,26642.0
7,2023-08-19 00:00:00+00:00,26051.0,26051.0,26051.0
8,2023-08-20 00:00:00+00:00,26097.0,26097.0,26097.0
9,2023-08-21 00:00:00+00:00,26192.0,26192.0,26192.0


,Forecast date,Previous actual,Manual naive,Frozen naive
1056,2026-07-03 00:00:00+00:00,61479.10,61479.10,61479.10
1057,2026-07-04 00:00:00+00:00,62522.46,62522.46,62522.46
1058,2026-07-05 00:00:00+00:00,63086.18,63086.18,63086.18
1059,2026-07-06 00:00:00+00:00,63587.06,63587.06,63587.06
1060,2026-07-07 00:00:00+00:00,64000.10,64000.10,64000.10


In [3]:
# Recompute the persistence errors directly.
error = test - naive
manual = {
    'MAE': np.abs(error).mean(),
    'RMSE': np.sqrt(np.mean(error ** 2)),
    'MAPE': 100 * np.mean(np.abs(error / test)),
    'sMAPE': 100 * np.mean(2 * np.abs(error) / (np.abs(test) + np.abs(naive))),
}

# Display the unchanged manual metrics.
manual

{'MAE': np.float64(1290.3532422243168),
 'RMSE': np.float64(1853.6247736716243),
 'MAPE': np.float64(1.742746521465369),
 'sMAPE': np.float64(1.7441417265023809)}

In [4]:
# Verify the implementation, rolling information set, and frozen vector.
checks = {
    'First forecast uses final training observation': np.isclose(naive.iloc[0], train.iloc[-1]),
    'Later forecasts use previous revealed actual': np.allclose(naive.iloc[1:], test.iloc[:-1]),
    'Frozen vector identity': np.allclose(naive, validated.Naive),
}
assert all(checks.values())

# Display the unchanged audit checks.
pd.Series(checks, name='Passed')

First forecast uses final training observation    True
Later forecasts use previous revealed actual      True
Frozen vector identity                            True
Name: Passed, dtype: bool

## Key Findings

The row audit confirms that the first naive forecast uses the final training observation and every later forecast uses only the immediately preceding revealed actual. The independently reconstructed vector is identical to the frozen naive vector, and the manually computed metrics therefore audit the persistence implementation directly.